# Project 07 — Robust Regression (Student-t vs Normal)

**Scenario.** A calibration experiment relates a known input $x$ to a measured response $y$ that should be linear, $y=\alpha+\beta x+\text{noise}$. Most points are clean, but a handful are **gross outliers** (a pipetting slip, a bubble, a mis-read plate). We want the calibration line without letting those few points drag it.

**New skill.** *Heavy tails, with $\nu$ as a parameter.* A Normal likelihood penalizes far points quadratically, so a single outlier can dominate the fit. The **Student-t** likelihood has heavy tails controlled by a degrees-of-freedom parameter $\nu$; small $\nu$ lets the model treat far points as tail events rather than evidence, making the fit robust. We fit **both** and let LOO confirm the Student-t wins.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240601

## Step 1 — Problem & data-generating story

Clean points follow $y=\alpha+\beta x+\text{Normal}(0,\sigma)$; a few are shifted by a large amount. The **known truth is the clean line** ($\alpha=1$, $\beta=2$, $\sigma=0.6$); a robust fit should recover it.

**Assumptions made explicit:** (a) the underlying relationship is linear; (b) most measurements are clean, a minority are gross outliers; (c) outliers are *vertical* (in $y$), not high-leverage corruptions of $x$; (d) inputs $x$ are known. The outliers here are placed at low-leverage (central $x$) with balanced signs, so they contaminate scatter without engineering a slope artefact.

In [ ]:
from data.generate_data import generate
data = generate()
x, y, out = data['x'], data['y'], data['is_outlier']
print(f"n={data['n']}, outliers={int(out.sum())}, "
      f"true alpha={data['truth']['alpha']}, beta={data['truth']['beta']}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x[~out], y[~out], color='#4C72B0', label='clean', s=25)
ax.scatter(x[out], y[out], color='#C44E52', label='outlier', s=45, marker='X')
tx = np.linspace(x.min(), x.max(), 50)
ax.plot(tx, data['truth']['alpha'] + data['truth']['beta']*tx, 'k--',
        label='true clean line')
ax.set(xlabel='x', ylabel='y', title='Calibration data with gross outliers')
ax.legend(); plt.tight_layout()

## Step 2 — Two likelihoods: Normal and Student-t

Both share $\mu_i=\alpha+\beta x_i$. They differ only in the noise model:

$$\text{Normal: } y_i\sim N(\mu_i,\sigma),\qquad\text{Student-t: } y_i\sim t_\nu(\mu_i,\sigma).$$

**Priors:** $\alpha,\beta\sim N(0,5)$, $\sigma\sim\text{HalfNormal}(5)$, and for the robust model $\nu\sim\text{Gamma}(2,0.1)$. The $\nu$ prior places real mass on small $\nu$ (heavy tails) but also lets $\nu$ grow if the data are clean — as $\nu\to\infty$ the Student-t becomes Normal, so the robust model nests the non-robust one.

In [ ]:
from model import build_model, fit
m_norm = build_model(data, model='normal')
m_t = build_model(data, model='studentt')
m_t

## Step 3 — Prior predictive check

We confirm the Student-t prior implies calibration lines of plausible slope and noisy-but-not-absurd responses. (The heavy tail means occasional large $y$ are *expected* under the prior — exactly the flexibility we want.)

In [ ]:
with m_t:
    prior = pm.sample_prior_predictive(draws=200, random_seed=RNG)
pp = prior.prior_predictive['y'].values.ravel()
pp = pp[np.abs(pp) < np.percentile(np.abs(pp), 99)]
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(pp, bins=40, color='#55A868', edgecolor='white')
ax.set(xlabel='prior-implied y (99th-pct clipped)', ylabel='draws',
       title='Student-t prior predictive — heavy-tailed but plausible')
plt.tight_layout()

## Step 4 — Inference (NUTS) for both models

Settings: `draws=1000, tune=1000, chains=4`. The Student-t can be slightly stiffer when $\nu$ is small, but with standardized $x$ default NUTS is fine. We keep both idatas for the comparison.

In [ ]:
idata_norm = fit(data, model='normal', draws=1000, tune=1000, chains=4, seed=101)
idata_t = fit(data, model='studentt', draws=1000, tune=1000, chains=4, seed=101)
print('normal divergences  :', int(idata_norm.sample_stats['diverging'].sum()))
print('studentt divergences:', int(idata_t.sample_stats['diverging'].sum()))

## Step 5 — Computational diagnostics

Both should converge ($\hat R\approx1$, healthy ESS). As with the Poisson in Project 06, the Normal model converges fine — it is *adequacy*, not convergence, that fails. The tell is in the estimates themselves.

In [ ]:
print('--- Normal ---')
print(az.summary(idata_norm, var_names=['alpha', 'beta', 'sigma']))
print('--- Student-t ---')
print(az.summary(idata_t, var_names=['alpha', 'beta', 'sigma', 'nu']))

**Read it now:** the Normal $\sigma$ is hugely inflated (~3 vs the clean 0.6 — it must 'explain' the outliers as ordinary noise), and as a direct consequence its slope/intercept become **far more uncertain** ($\beta$ has an SD ~6× the Student-t's). Because these outliers are balanced and low-leverage by design, the Normal's slope *mean* is not strongly biased — the damage is to its precision and its noise estimate. The Student-t recovers $\alpha\approx1$, $\beta\approx2$ with tight intervals, a small scale, and a small $\nu$ (heavy tails) — it has *identified* the outliers as tail events.

## Step 6 — Posterior predictive & the fitted lines

Overlay both fitted lines on the data. The Normal fit is far less certain (a much wider posterior band, driven by its inflated $\sigma$); the Student-t line tracks the clean trend tightly. With these balanced, low-leverage outliers the Normal *mean* line is close to the truth too — the cost of ignoring robustness shows up as ballooning uncertainty and a 5× inflated noise scale rather than a tilted mean line.

In [ ]:
tx = np.linspace(x.min(), x.max(), 50)
def line(idata):
    a = idata.posterior['alpha'].values.ravel()
    b = idata.posterior['beta'].values.ravel()
    yy = a[:, None] + b[:, None] * tx[None, :]
    return yy.mean(0), np.percentile(yy, [3, 97], axis=0)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(x[~out], y[~out], color='#4C72B0', s=25, label='clean')
ax.scatter(x[out], y[out], color='#C44E52', s=45, marker='X', label='outlier')
mn, _ = line(idata_norm); mt, _ = line(idata_t)
ax.plot(tx, mn, color='#C44E52', lw=2, label='Normal fit (noisy/uncertain)')
ax.plot(tx, mt, color='#55A868', lw=2, label='Student-t fit (robust)')
ax.plot(tx, data['truth']['alpha']+data['truth']['beta']*tx, 'k--', label='truth')
ax.set(xlabel='x', ylabel='y', title='Normal destabilized by outliers; Student-t robust')
ax.legend(); plt.tight_layout()

In [ ]:
az.plot_ppc(idata_t, num_pp_samples=100)
plt.title('Student-t posterior predictive'); plt.tight_layout()

## Step 7 — Model comparison (LOO) & recovery

PSIS-LOO should prefer the Student-t decisively: down-weighting outliers yields better held-out prediction. We then confirm recovery of the clean line by the robust model.

In [ ]:
cmp = az.compare({'normal': idata_norm, 'studentt': idata_t}, ic='loo')
print(cmp[['rank', 'elpd_loo', 'p_loo', 'elpd_diff', 'dse', 'weight']])

In [ ]:
from shared.bayes_utils import check_recovery
line_truth = {k: data['truth'][k] for k in ('alpha', 'beta')}
print('Student-t recovery of the clean line:')
for res in check_recovery(idata_t, line_truth):
    print(res)
print('\nNormal estimate of the slope (for contrast):')
print(az.summary(idata_norm, var_names=['beta'])[['mean', 'sd']])

## Step 8 — Decision & communication

Report the calibration slope and intercept with intervals, and state that the robust model was used so the few bad wells did not distort the result.

In [ ]:
b = idata_t.posterior['beta'].values.ravel()
a = idata_t.posterior['alpha'].values.ravel()
print(f'Robust slope  beta : {b.mean():.3f}  94% [{np.percentile(b,3):.3f}, {np.percentile(b,97):.3f}]')
print(f'Robust intercept   : {a.mean():.3f}  94% [{np.percentile(a,3):.3f}, {np.percentile(a,97):.3f}]')
nu = idata_t.posterior['nu'].values.ravel()
print(f'Estimated nu (tail weight): {nu.mean():.2f} (small => heavy tails => outliers present)')

**Conclusion (for a collaborator).** The calibration line is $y\approx 1.0 + 2.0\,x$, recovered cleanly despite ~10% gross outliers. A small $\nu$ tells us the outliers are real and were correctly down-weighted. A naive least-squares / Normal fit would have reported a wildly inflated noise level and a much less certain line (here ~6× the slope SD). See `summary_onepager.md`.